In [11]:
# Importação de bibliotecas e inciar o Spark

import os
from pyspark.sql import SparkSession
import pandas as pd
import pyodbc
from pyspark.sql.functions import count, avg, col

spark = (
    SparkSession
    .builder
    .appName("ETL-Netflix-03-Load")
    .config("spark.driver.memory", "2g")
    .getOrCreate()
)

print("Bibliotecas importadas e Spark iniciado")

Bibliotecas importadas e Spark iniciado


In [12]:
# Definindo diretórios

base_path = "../input"
silver_path = os.path.join(base_path, "silver", "netflix-silver")
gold_path = os.path.join(base_path, "gold")

os.makedirs(gold_path, exist_ok=True)

# Configurações JDBC
jdbc_url = "jdbc:sqlserver://localhost:1434;databaseName=datalake_local;encrypt=true;trustServerCertificate=true"
jdbc_properties = { 
    "user": "sa", 
    "password": "Pipocando@2", 
    "driver": "com.microsoft.sqlserver.jdbc.SQLServerDriver" 
}

print("Diretórios definidos")

Diretórios definidos


In [13]:
# Ler os dados da camada silver

df_silver = spark.read.option("header", True).csv(silver_path)
print("Dados carregados da camada Silver:")
df_silver.show(5)

Dados carregados da camada Silver:
+-------+-------+--------------------+---------------+--------------------+--------------+------------------+------------+------+------------+--------------------+--------------------+--------------+
|show_id|   type|               title|       director|                cast|       country|        date_added|release_year|rating|duration_raw|               genre|         description|duration_value|
+-------+-------+--------------------+---------------+--------------------+--------------+------------------+------------+------+------------+--------------------+--------------------+--------------+
|     s1|  Movie|Dick Johnson Is Dead|Kirsten Johnson|                NULL| United States|September 25, 2021|        2020| PG-13|      90 min|       Documentaries|As her father nea...|            90|
|     s2|TV Show|       Blood & Water|           NULL|Ama Qamata, Khosi...|  South Africa|September 24, 2021|        2021| TV-MA|   2 Seasons|International TV ...|Af

In [14]:
# Criar com o Dataset Gold com Agregações

df_gold = (
    df_silver
    .groupBy("country", "type")
    .agg(
        count("*").alias("total_titles"),
        avg("duration_value").alias("avg_duration")
    )
    .orderBy(col("total_titles").desc())
)

print("Dataset Gold criado:")
df_gold.show(10)

Dataset Gold criado:
+--------------+-------+------------+------------------+
|       country|   type|total_titles|      avg_duration|
+--------------+-------+------------+------------------+
| United States|  Movie|        2473| 92.43886639676113|
| United States|TV Show|         904|2.2964601769911503|
|         India|  Movie|         876|123.16666666666667|
|United Kingdom|  Movie|         495| 98.03434343434344|
|        Canada|  Movie|         314| 91.30891719745223|
|        France|  Movie|         282|100.50709219858156|
|United Kingdom|TV Show|         266|1.8421052631578947|
|         Japan|TV Show|         188| 1.574468085106383|
|       Germany|  Movie|         172|102.22674418604652|
|   South Korea|TV Show|         170|1.2529411764705882|
+--------------+-------+------------+------------------+
only showing top 10 rows


In [15]:
# Salvar CSV da camada gold

gold_output_path = os.path.join(gold_path, "netflix-gold")
df_gold.write.mode("overwrite").option("header", True).csv(gold_output_path)

print(f"Arquivo Gold salvo com sucesso em: {gold_output_path}")

Arquivo Gold salvo com sucesso em: ../input\gold\netflix-gold


In [16]:
# Inserir no SQL com Inserts Dinâmicospdf = df_gold.toPandas()

conn_str = (
    "DRIVER={ODBC Driver 17 for SQL Server};"
    "SERVER=localhost,1434;"
    "DATABASE=datalake_local;"
    "UID=sa;"
    "PWD=Pipocando@2;"
    "TrustServerCertificate=yes;"
)
conn = pyodbc.connect(conn_str)
cursor = conn.cursor()

cursor.execute("""
IF NOT EXISTS (SELECT * FROM sysobjects WHERE name='netflix_gold' AND xtype='U')
CREATE TABLE netflix_gold (
    country NVARCHAR(255),
    type NVARCHAR(50),
    total_titles INT,
    avg_duration FLOAT
)
""")
conn.commit()

for _, row in pdf.iterrows():
    cursor.execute("""
        INSERT INTO netflix_gold (country, type, total_titles, avg_duration)
        VALUES (?, ?, ?, ?)
    """, row['country'], row['type'], int(row['total_titles']), float(row['avg_duration']) if row['avg_duration'] else None)

conn.commit()
cursor.close()
conn.close()

print("Dados inseridos no SQL Server com sucesso (tabela: netflix_gold).")

OperationalError: ('08001', '[08001] [Microsoft][ODBC Driver 17 for SQL Server]Provedor TCP: Nenhuma conexão pôde ser feita porque a máquina de destino as recusou ativamente.\r\n (10061) (SQLDriverConnect); [08001] [Microsoft][ODBC Driver 17 for SQL Server]O tempo limite do logon expirou (0); [08001] [Microsoft][ODBC Driver 17 for SQL Server]Erro relatado pela rede ou específico à instância ao estabelecer conexão com o SQL Server. O servidor não foi encontrado ou não está acessível. Verifique se o nome da instância está correto e se o SQL Server está configurado para permitir conexões remotas. Para obter mais informações, consulte os Manuais Online do SQL Server. (10061)')